In [1]:
print("Python is working")

Python is working


In [2]:
# ============================================================
# ANVAYA SAṂHATI
# Hospital Intelligence ML Pipeline
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

warnings.filterwarnings("ignore")

print("ANVAYA ML environment loaded successfully.")

ANVAYA ML environment loaded successfully.


In [3]:
# ============================================================
# ANVAYA ML PROJECT STRUCTURE
# ============================================================

ML_ROOT = Path(r"B:\VS CODE\TY\EDAI\TRAINED MODEL")

DATA_RAW = ML_ROOT / "data" / "raw"
DATA_PROCESSED = ML_ROOT / "data" / "processed"
MODELS = ML_ROOT / "models"
REPORTS = ML_ROOT / "reports"
FIGURES = REPORTS / "figures"
METRICS = REPORTS / "metrics"

for folder in [
    DATA_RAW,
    DATA_PROCESSED,
    MODELS,
    REPORTS,
    FIGURES,
    METRICS
]:
    folder.mkdir(parents=True, exist_ok=True)

print("ML project structure ready.")
print()
print("Root:", ML_ROOT)
print("Raw data:", DATA_RAW)
print("Processed data:", DATA_PROCESSED)
print("Models:", MODELS)
print("Reports:", REPORTS)

ML project structure ready.

Root: B:\VS CODE\TY\EDAI\TRAINED MODEL
Raw data: B:\VS CODE\TY\EDAI\TRAINED MODEL\data\raw
Processed data: B:\VS CODE\TY\EDAI\TRAINED MODEL\data\processed
Models: B:\VS CODE\TY\EDAI\TRAINED MODEL\models
Reports: B:\VS CODE\TY\EDAI\TRAINED MODEL\reports


In [4]:
# ============================================================
# ANVAYA SAṂHATI
# SYNTHETIC HOSPITAL DATASET
# ============================================================

RANDOM_STATE = 42
N_SAMPLES = 50_000

rng = np.random.default_rng(RANDOM_STATE)

print("Generating hospital dataset...")
print(f"Records: {N_SAMPLES:,}")

# ------------------------------------------------------------
# PATIENT DEMOGRAPHICS
# ------------------------------------------------------------

patient_id = [
    f"ANV-P-{i:06d}"
    for i in range(1, N_SAMPLES + 1)
]

age = rng.integers(
    18,
    91,
    N_SAMPLES
)

gender = rng.choice(
    ["Male", "Female"],
    size=N_SAMPLES,
    p=[0.55, 0.45]
)

# ------------------------------------------------------------
# HOSPITAL DEPARTMENT
# ------------------------------------------------------------

departments = [
    "Cardiology",
    "General Medicine",
    "Orthopedics",
    "Neurology",
    "Pulmonology",
    "Emergency",
    "General Surgery"
]

department = rng.choice(
    departments,
    size=N_SAMPLES
)

# ------------------------------------------------------------
# ENCOUNTER TYPE
# ------------------------------------------------------------

encounter_type = rng.choice(
    [
        "OPD",
        "IPD",
        "EMERGENCY",
        "OBSERVATION"
    ],
    size=N_SAMPLES,
    p=[
        0.35,
        0.30,
        0.20,
        0.15
    ]
)

# ------------------------------------------------------------
# VITAL SIGNS
# ------------------------------------------------------------

heart_rate = np.clip(
    rng.normal(82, 15, N_SAMPLES),
    45,
    150
)

systolic_bp = np.clip(
    rng.normal(125, 20, N_SAMPLES),
    80,
    200
)

diastolic_bp = np.clip(
    rng.normal(80, 12, N_SAMPLES),
    45,
    130
)

temperature = np.clip(
    rng.normal(98.4, 1.2, N_SAMPLES),
    95,
    105
)

spo2 = np.clip(
    rng.normal(96.5, 2.5, N_SAMPLES),
    80,
    100
)

respiratory_rate = np.clip(
    rng.normal(18, 4, N_SAMPLES),
    8,
    40
)

# ------------------------------------------------------------
# MEDICAL CONDITIONS
# ------------------------------------------------------------

diabetes_probability = np.clip(
    0.08 + age * 0.003,
    0.08,
    0.45
)

hypertension_probability = np.clip(
    0.12 + age * 0.004,
    0.12,
    0.55
)

cardiac_probability = np.clip(
    0.03 + age * 0.0025,
    0.03,
    0.30
)

respiratory_probability = np.clip(
    0.04 + age * 0.0015,
    0.04,
    0.20
)

diabetes = rng.binomial(
    1,
    diabetes_probability
)

hypertension = rng.binomial(
    1,
    hypertension_probability
)

cardiac_condition = rng.binomial(
    1,
    cardiac_probability
)

respiratory_condition = rng.binomial(
    1,
    respiratory_probability
)

# ------------------------------------------------------------
# PREVIOUS MEDICAL HISTORY
# ------------------------------------------------------------

previous_admissions = np.clip(
    rng.poisson(
        0.5 + age / 100
    ),
    0,
    8
)

previous_emergency_visits = np.clip(
    rng.poisson(
        0.4 + age / 120
    ),
    0,
    7
)

medication_count = np.clip(
    rng.poisson(
        2 + age / 30
    ),
    0,
    15
)

lab_abnormal_count = np.clip(
    rng.poisson(
        1.2,
        N_SAMPLES
    ),
    0,
    10
)

# ------------------------------------------------------------
# ICU INDICATOR
# ------------------------------------------------------------

icu_probability = np.where(
    encounter_type == "EMERGENCY",
    0.18,
    0.04
)

icu_flag = rng.binomial(
    1,
    icu_probability
)

# ------------------------------------------------------------
# CREATE DATAFRAME
# ------------------------------------------------------------

df = pd.DataFrame({
    "patient_id": patient_id,
    "age": age,
    "gender": gender,
    "department": department,
    "encounter_type": encounter_type,

    "heart_rate": heart_rate.round(1),
    "systolic_bp": systolic_bp.round(1),
    "diastolic_bp": diastolic_bp.round(1),
    "temperature": temperature.round(2),
    "spo2": spo2.round(1),
    "respiratory_rate": respiratory_rate.round(1),

    "diabetes": diabetes,
    "hypertension": hypertension,
    "cardiac_condition": cardiac_condition,
    "respiratory_condition": respiratory_condition,

    "previous_admissions": previous_admissions,
    "previous_emergency_visits": previous_emergency_visits,
    "medication_count": medication_count,
    "lab_abnormal_count": lab_abnormal_count,
    "icu_flag": icu_flag
})

print("\nDataset generated successfully.")
print("Shape:", df.shape)

display(df.head(10))

Generating hospital dataset...
Records: 50,000

Dataset generated successfully.
Shape: (50000, 20)


,patient_id,age,gender,department,encounter_type,heart_rate,systolic_bp,diastolic_bp,temperature,spo2,respiratory_rate,diabetes,hypertension,cardiac_condition,respiratory_condition,previous_admissions,previous_emergency_visits,medication_count,lab_abnormal_count,icu_flag
0,ANV-P-000001,24,Female,General Medicine,OBSERVATION,101.1,133.0,79.2,97.17,96.6,29.8,0,1,0,0,1,0,0,0,0
1,ANV-P-000002,74,Female,General Surgery,OBSERVATION,83.5,137.9,58.8,98.87,99.1,20.5,0,0,0,0,3,1,5,3,0
2,ANV-P-000003,65,Female,Cardiology,IPD,81.7,118.4,83.3,97.70,99.4,12.7,0,0,0,0,1,2,2,1,0
3,ANV-P-000004,50,Male,Orthopedics,OPD,97.3,143.7,83.3,98.24,98.0,14.4,0,0,0,0,1,0,3,2,0
4,ANV-P-000005,49,Male,Neurology,EMERGENCY,108.4,145.0,73.4,96.80,91.3,20.0,0,0,0,0,2,0,5,0,0
5,ANV-P-000006,80,Male,General Surgery,EMERGENCY,75.7,114.2,84.3,99.24,98.2,20.3,0,0,0,0,2,1,5,0,0
6,ANV-P-000007,24,Male,Emergency,OBSERVATION,82.7,116.1,61.2,98.42,96.9,21.3,0,0,0,0,1,0,4,3,0
7,ANV-P-000008,68,Female,General Surgery,OBSERVATION,83.9,116.5,103.2,98.04,92.6,23.7,0,0,0,0,2,1,3,0,0
8,ANV-P-000009,32,Female,Neurology,OPD,79.4,122.8,64.7,98.87,95.0,11.1,0,1,0,0,2,0,1,0,0
9,ANV-P-000010,24,Female,Emergency,IPD,84.1,114.6,87.1,97.93,97.1,12.0,0,0,0,0,1,1,1,1,0


In [5]:
# ============================================================
# ANVAYA SAṂHATI
# TARGET GENERATION
# ============================================================

print("Generating ML targets...")

# ------------------------------------------------------------
# 1. CLINICAL SEVERITY SCORE
# ------------------------------------------------------------

severity_score = (
    0.018 * age
    + 0.035 * np.abs(heart_rate - 75)
    + 0.025 * np.maximum(systolic_bp - 140, 0)
    + 0.80 * np.maximum(92 - spo2, 0)
    + 0.50 * np.maximum(temperature - 100.0, 0)
    + 0.40 * respiratory_condition
    + 0.50 * cardiac_condition
    + 0.25 * diabetes
    + 0.25 * hypertension
    + 0.30 * previous_admissions
    + 0.25 * previous_emergency_visits
    + 0.15 * medication_count
    + 0.35 * lab_abnormal_count
    + 1.20 * icu_flag
)

# ------------------------------------------------------------
# 2. LENGTH OF STAY
# ------------------------------------------------------------

los_base = (
    1.5
    + 0.025 * age
    + 0.8 * diabetes
    + 1.0 * hypertension
    + 1.5 * cardiac_condition
    + 1.2 * respiratory_condition
    + 0.35 * previous_admissions
    + 0.45 * lab_abnormal_count
    + 2.5 * icu_flag
)

los_noise = rng.normal(
    0,
    1.5,
    N_SAMPLES
)

length_of_stay = np.clip(
    los_base + los_noise,
    1,
    30
)

# ------------------------------------------------------------
# 3. READMISSION WITHIN 30 DAYS
# ------------------------------------------------------------

readmission_score = (
    -3.2
    + 0.018 * age
    + 0.35 * diabetes
    + 0.45 * cardiac_condition
    + 0.25 * hypertension
    + 0.20 * respiratory_condition
    + 0.30 * previous_admissions
    + 0.35 * previous_emergency_visits
    + 0.15 * lab_abnormal_count
    + 0.08 * medication_count
)

readmission_probability = (
    1 / (1 + np.exp(-readmission_score))
)

readmitted_30d = rng.binomial(
    1,
    readmission_probability
)

# ------------------------------------------------------------
# 4. HIGH-RISK PATIENT
# ------------------------------------------------------------

risk_probability = (
    1 / (1 + np.exp(-(severity_score - 7)))
)

high_risk = rng.binomial(
    1,
    risk_probability
)

# ------------------------------------------------------------
# 5. CLINICAL DETERIORATION
# ------------------------------------------------------------

deterioration_score = (
    -5
    + 0.06 * np.abs(heart_rate - 80)
    + 0.10 * np.maximum(95 - spo2, 0)
    + 0.50 * np.maximum(temperature - 100, 0)
    + 0.08 * np.maximum(respiratory_rate - 20, 0)
    + 0.025 * np.maximum(140 - systolic_bp, 0)
    + 0.8 * cardiac_condition
    + 0.7 * respiratory_condition
    + 0.6 * icu_flag
)

deterioration_probability = (
    1 / (1 + np.exp(-deterioration_score))
)

deterioration = rng.binomial(
    1,
    deterioration_probability
)

# ------------------------------------------------------------
# ADD TARGETS TO DATAFRAME
# ------------------------------------------------------------

df["length_of_stay"] = length_of_stay.round(1)
df["readmitted_30d"] = readmitted_30d
df["high_risk"] = high_risk
df["deterioration"] = deterioration

print("Targets generated successfully.")

print("\nTarget summary:")
print("--------------------------------------------")

print(
    f"High-risk patients       : "
    f"{df['high_risk'].sum():,} "
    f"({df['high_risk'].mean()*100:.2f}%)"
)

print(
    f"30-day readmissions      : "
    f"{df['readmitted_30d'].sum():,} "
    f"({df['readmitted_30d'].mean()*100:.2f}%)"
)

print(
    f"Deterioration cases      : "
    f"{df['deterioration'].sum():,} "
    f"({df['deterioration'].mean()*100:.2f}%)"
)

print(
    f"Average length of stay   : "
    f"{df['length_of_stay'].mean():.2f} days"
)

print("--------------------------------------------")

Generating ML targets...
Targets generated successfully.

Target summary:
--------------------------------------------
High-risk patients       : 2,328 (4.66%)
30-day readmissions      : 15,861 (31.72%)
Deterioration cases      : 2,078 (4.16%)
Average length of stay   : 4.87 days
--------------------------------------------


In [6]:
# ============================================================
# DATA QUALITY CHECK
# ============================================================

print("DATASET QUALITY REPORT")
print("=" * 60)

print(f"Total records       : {len(df):,}")
print(f"Total features      : {len(df.columns)}")

print("\nMissing values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values")

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nNumerical summary:")
display(df.describe().T)

DATASET QUALITY REPORT
Total records       : 50,000
Total features      : 24

Missing values:
No missing values

Duplicate rows:
0

Data types:
patient_id                    object
age                            int64
gender                        object
department                    object
encounter_type                object
heart_rate                   float64
systolic_bp                  float64
diastolic_bp                 float64
temperature                  float64
spo2                         float64
respiratory_rate             float64
diabetes                       int64
hypertension                   int64
cardiac_condition              int64
respiratory_condition          int64
previous_admissions            int64
previous_emergency_visits      int64
medication_count               int64
lab_abnormal_count             int64
icu_flag                       int64
length_of_stay               float64
readmitted_30d                 int64
high_risk                      int64
deter

,count,mean,std,min,25%,50%,75%,max
age,50000.0,53.958380,21.090943,18.0,36.0,54.0,72.0,90.00
heart_rate,50000.0,82.121370,14.902212,45.0,72.0,82.1,92.2,150.00
systolic_bp,50000.0,125.142740,19.689441,80.0,111.7,125.2,138.5,200.00
diastolic_bp,50000.0,80.020722,11.972328,45.0,71.9,80.0,88.1,130.00
temperature,50000.0,98.399385,1.194202,95.0,97.6,98.4,99.2,103.31
spo2,50000.0,96.405442,2.330973,85.9,94.8,96.5,98.2,100.00
respiratory_rate,50000.0,17.977014,3.992804,8.0,15.3,18.0,20.7,35.80
diabetes,50000.0,0.239040,0.426501,0.0,0.0,0.0,0.0,1.00
hypertension,50000.0,0.335960,0.472330,0.0,0.0,0.0,1.0,1.00
cardiac_condition,50000.0,0.162880,0.369260,0.0,0.0,0.0,0.0,1.00


In [7]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================

# Features used by the ML models
FEATURE_COLUMNS = [
    "age",
    "gender",
    "department",
    "encounter_type",
    "heart_rate",
    "systolic_bp",
    "diastolic_bp",
    "temperature",
    "spo2",
    "respiratory_rate",
    "diabetes",
    "hypertension",
    "cardiac_condition",
    "respiratory_condition",
    "previous_admissions",
    "previous_emergency_visits",
    "medication_count",
    "lab_abnormal_count",
    "icu_flag"
]

X = df[FEATURE_COLUMNS].copy()

print("Feature matrix created.")
print("Rows:", X.shape[0])
print("Features:", X.shape[1])

display(X.head())

Feature matrix created.
Rows: 50000
Features: 19


,age,gender,department,encounter_type,heart_rate,systolic_bp,diastolic_bp,temperature,spo2,respiratory_rate,diabetes,hypertension,cardiac_condition,respiratory_condition,previous_admissions,previous_emergency_visits,medication_count,lab_abnormal_count,icu_flag
0,24,Female,General Medicine,OBSERVATION,101.1,133.0,79.2,97.17,96.6,29.8,0,1,0,0,1,0,0,0,0
1,74,Female,General Surgery,OBSERVATION,83.5,137.9,58.8,98.87,99.1,20.5,0,0,0,0,3,1,5,3,0
2,65,Female,Cardiology,IPD,81.7,118.4,83.3,97.70,99.4,12.7,0,0,0,0,1,2,2,1,0
3,50,Male,Orthopedics,OPD,97.3,143.7,83.3,98.24,98.0,14.4,0,0,0,0,1,0,3,2,0
4,49,Male,Neurology,EMERGENCY,108.4,145.0,73.4,96.80,91.3,20.0,0,0,0,0,2,0,5,0,0


In [8]:
# ============================================================
# CATEGORICAL ENCODING
# ============================================================

X_encoded = pd.get_dummies(
    X,
    columns=[
        "gender",
        "department",
        "encounter_type"
    ],
    dtype=int
)

print("Categorical encoding completed.")
print("Original features :", X.shape[1])
print("Encoded features  :", X_encoded.shape[1])

display(X_encoded.head())

Categorical encoding completed.
Original features : 19
Encoded features  : 29


,age,heart_rate,systolic_bp,diastolic_bp,temperature,spo2,respiratory_rate,diabetes,hypertension,cardiac_condition,...,department_Emergency,department_General Medicine,department_General Surgery,department_Neurology,department_Orthopedics,department_Pulmonology,encounter_type_EMERGENCY,encounter_type_IPD,encounter_type_OBSERVATION,encounter_type_OPD
0,24,101.1,133.0,79.2,97.17,96.6,29.8,0,1,0,...,0,1,0,0,0,0,0,0,1,0
1,74,83.5,137.9,58.8,98.87,99.1,20.5,0,0,0,...,0,0,1,0,0,0,0,0,1,0
2,65,81.7,118.4,83.3,97.70,99.4,12.7,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,50,97.3,143.7,83.3,98.24,98.0,14.4,0,0,0,...,0,0,0,0,1,0,0,0,0,1
4,49,108.4,145.0,73.4,96.80,91.3,20.0,0,0,0,...,0,0,0,1,0,0,1,0,0,0


In [9]:
# ============================================================
# MODEL 1 — PATIENT RISK PREDICTION
# ============================================================

TARGET = "high_risk"

y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Patient Risk Dataset")
print("=" * 50)
print(f"Training records : {len(X_train):,}")
print(f"Testing records  : {len(X_test):,}")
print(f"Features         : {X_train.shape[1]}")
print(f"Positive cases   : {y.sum():,}")

Patient Risk Dataset
Training records : 40,000
Testing records  : 10,000
Features         : 29
Positive cases   : 2,328


In [10]:
# ============================================================
# MODEL 1 — TRAIN PATIENT RISK MODEL
# ============================================================

print("Training Patient Risk model...")
print("=" * 60)

patient_risk_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

patient_risk_model.fit(
    X_train,
    y_train
)

print("Training completed successfully.")

Training Patient Risk model...
Training completed successfully.


In [11]:
# ============================================================
# MODEL 1 — PATIENT RISK EVALUATION
# ============================================================

print("Evaluating Patient Risk model...")
print("=" * 60)

# Predictions
risk_predictions = patient_risk_model.predict(X_test)

# Probability of HIGH RISK
risk_probabilities = patient_risk_model.predict_proba(X_test)[:, 1]

# Metrics
risk_accuracy = accuracy_score(
    y_test,
    risk_predictions
)

risk_auc = roc_auc_score(
    y_test,
    risk_probabilities
)

print(f"Accuracy : {risk_accuracy:.4f}")
print(f"ROC-AUC  : {risk_auc:.4f}")

print("\nClassification Report")
print("-" * 60)

print(
    classification_report(
        y_test,
        risk_predictions,
        target_names=[
            "Low Risk",
            "High Risk"
        ],
        digits=4
    )
)

print("\nConfusion Matrix")
print("-" * 60)

risk_cm = confusion_matrix(
    y_test,
    risk_predictions
)

print(risk_cm)

Evaluating Patient Risk model...
Accuracy : 0.9358
ROC-AUC  : 0.7475

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Low Risk     0.9606    0.9725    0.9665      9534
   High Risk     0.2471    0.1845    0.2113       466

    accuracy                         0.9358     10000
   macro avg     0.6039    0.5785    0.5889     10000
weighted avg     0.9274    0.9358    0.9313     10000


Confusion Matrix
------------------------------------------------------------
[[9272  262]
 [ 380   86]]


In [12]:
# ============================================================
# MODEL 1 — FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": patient_risk_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("Top features learned by Patient Risk model:")
display(feature_importance.head(15))

Top features learned by Patient Risk model:


,feature,importance
0,age,0.170321
1,heart_rate,0.094154
2,systolic_bp,0.076416
13,medication_count,0.076129
5,spo2,0.073863
6,respiratory_rate,0.063860
4,temperature,0.063711
3,diastolic_bp,0.063355
11,previous_admissions,0.054089
14,lab_abnormal_count,0.052331


In [13]:
# ============================================================
# MODEL 1B — TUNED PATIENT RISK MODEL
# ============================================================

print("Training tuned Patient Risk model...")
print("=" * 60)

patient_risk_model_v2 = RandomForestClassifier(
    n_estimators=500,
    max_depth=16,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

patient_risk_model_v2.fit(
    X_train,
    y_train
)

print("Tuned model training completed.")

Training tuned Patient Risk model...
Tuned model training completed.


In [14]:
# ============================================================
# MODEL 1B — EVALUATION
# ============================================================

risk_v2_predictions = patient_risk_model_v2.predict(X_test)

risk_v2_probabilities = (
    patient_risk_model_v2.predict_proba(X_test)[:, 1]
)

risk_v2_accuracy = accuracy_score(
    y_test,
    risk_v2_predictions
)

risk_v2_auc = roc_auc_score(
    y_test,
    risk_v2_probabilities
)

print("TUNED PATIENT RISK MODEL")
print("=" * 60)

print(f"Accuracy : {risk_v2_accuracy:.4f}")
print(f"ROC-AUC  : {risk_v2_auc:.4f}")

print("\nClassification Report")
print("-" * 60)

print(
    classification_report(
        y_test,
        risk_v2_predictions,
        target_names=["Low Risk", "High Risk"],
        digits=4
    )
)

print("\nConfusion Matrix")
print("-" * 60)

print(
    confusion_matrix(
        y_test,
        risk_v2_predictions
    )
)

TUNED PATIENT RISK MODEL
Accuracy : 0.9531
ROC-AUC  : 0.7471

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Low Risk     0.9535    0.9996    0.9760      9534
   High Risk     0.2000    0.0021    0.0042       466

    accuracy                         0.9531     10000
   macro avg     0.5767    0.5009    0.4901     10000
weighted avg     0.9184    0.9531    0.9307     10000


Confusion Matrix
------------------------------------------------------------
[[9530    4]
 [ 465    1]]


In [15]:
# ============================================================
# MODEL 1 — VALIDATION SET FOR THRESHOLD TUNING
# ============================================================

X_train_main, X_val, y_train_main, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train
)

print("Patient Risk validation split")
print("=" * 60)
print(f"Training records   : {len(X_train_main):,}")
print(f"Validation records : {len(X_val):,}")
print(f"Test records       : {len(X_test):,}")

Patient Risk validation split
Training records   : 32,000
Validation records : 8,000
Test records       : 10,000


In [16]:
# ============================================================
# MODEL 1C — PATIENT RISK MODEL FOR THRESHOLD TUNING
# ============================================================

patient_risk_model_v3 = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

patient_risk_model_v3.fit(
    X_train_main,
    y_train_main
)

print("Model 1C training completed.")

Model 1C training completed.


In [17]:
# ============================================================
# MODEL 1C — THRESHOLD ANALYSIS
# ============================================================

from sklearn.metrics import precision_recall_curve, f1_score

# Validation probabilities
val_probabilities = (
    patient_risk_model_v3.predict_proba(X_val)[:, 1]
)

# Test a range of thresholds
threshold_results = []

for threshold in np.arange(0.10, 0.71, 0.01):

    val_predictions = (
        val_probabilities >= threshold
    ).astype(int)

    precision = (
        np.sum((val_predictions == 1) & (y_val == 1))
        / max(np.sum(val_predictions == 1), 1)
    )

    recall = (
        np.sum((val_predictions == 1) & (y_val == 1))
        / max(np.sum(y_val == 1), 1)
    )

    f1 = f1_score(
        y_val,
        val_predictions,
        zero_division=0
    )

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(threshold_results)

# Select threshold with highest F1
best_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

BEST_THRESHOLD = float(
    best_row["threshold"]
)

print("THRESHOLD ANALYSIS")
print("=" * 60)

print(f"Best threshold : {BEST_THRESHOLD:.2f}")
print(f"Precision      : {best_row['precision']:.4f}")
print(f"Recall         : {best_row['recall']:.4f}")
print(f"F1 Score       : {best_row['f1']:.4f}")

print("\nTop threshold candidates:")
display(
    threshold_df
    .sort_values("f1", ascending=False)
    .head(10)
)

THRESHOLD ANALYSIS
Best threshold : 0.42
Precision      : 0.1402
Recall         : 0.3091
F1 Score       : 0.1930

Top threshold candidates:


,threshold,precision,recall,f1
32,0.42,0.140244,0.309140,0.192953
31,0.41,0.133043,0.327957,0.189294
30,0.40,0.126706,0.349462,0.185980
33,0.43,0.140669,0.271505,0.185321
29,0.39,0.121739,0.376344,0.183968
34,0.44,0.146538,0.244624,0.183283
28,0.38,0.117693,0.405914,0.182477
35,0.45,0.152416,0.220430,0.180220
27,0.37,0.112296,0.424731,0.177628
36,0.46,0.157895,0.201613,0.177096


In [18]:
# ============================================================
# MODEL 1C — FINAL TEST EVALUATION
# ============================================================

print("FINAL PATIENT RISK MODEL EVALUATION")
print("=" * 60)

# Test-set probabilities
test_probabilities = (
    patient_risk_model_v3.predict_proba(X_test)[:, 1]
)

# Apply threshold selected ONLY from validation data
test_predictions = (
    test_probabilities >= BEST_THRESHOLD
).astype(int)

# Metrics
final_accuracy = accuracy_score(
    y_test,
    test_predictions
)

final_auc = roc_auc_score(
    y_test,
    test_probabilities
)

print(f"Decision threshold : {BEST_THRESHOLD:.2f}")
print(f"Accuracy           : {final_accuracy:.4f}")
print(f"ROC-AUC            : {final_auc:.4f}")

print("\nClassification Report")
print("-" * 60)

print(
    classification_report(
        y_test,
        test_predictions,
        target_names=[
            "Low Risk",
            "High Risk"
        ],
        digits=4
    )
)

print("\nConfusion Matrix")
print("-" * 60)

final_cm = confusion_matrix(
    y_test,
    test_predictions
)

print(final_cm)

FINAL PATIENT RISK MODEL EVALUATION
Decision threshold : 0.42
Accuracy           : 0.8845
ROC-AUC            : 0.7433

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Low Risk     0.9680    0.9089    0.9375      9534
   High Risk     0.1716    0.3863    0.2376       466

    accuracy                         0.8845     10000
   macro avg     0.5698    0.6476    0.5876     10000
weighted avg     0.9309    0.8845    0.9049     10000


Confusion Matrix
------------------------------------------------------------
[[8665  869]
 [ 286  180]]


In [19]:
# ============================================================
# SAVE MODEL 1 — PATIENT RISK
# ============================================================

PATIENT_RISK_DIR = MODELS / "patient_risk"
PATIENT_RISK_DIR.mkdir(parents=True, exist_ok=True)

# Save trained model
patient_risk_model_path = (
    PATIENT_RISK_DIR / "patient_risk_model.joblib"
)

joblib.dump(
    patient_risk_model_v3,
    patient_risk_model_path
)

# Save feature schema + threshold + metrics
patient_risk_metadata = {
    "model_name": "ANVAYA Patient Risk Model",
    "model_type": "RandomForestClassifier",
    "version": "1.0",
    "target": "high_risk",
    "decision_threshold": BEST_THRESHOLD,
    "features": list(X_encoded.columns),
    "training_records": int(len(X_train_main)),
    "validation_records": int(len(X_val)),
    "test_records": int(len(X_test)),
    "metrics": {
        "accuracy": round(final_accuracy, 4),
        "roc_auc": round(final_auc, 4),
        "high_risk_precision": 0.1716,
        "high_risk_recall": 0.3863,
        "high_risk_f1": 0.2376
    },
    "data_type": "synthetic_demo_data",
    "purpose": "Academic hospital risk-ranking prototype"
}

metadata_path = (
    PATIENT_RISK_DIR / "metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        patient_risk_metadata,
        f,
        indent=4
    )

print("Patient Risk model saved successfully.")
print()
print("Model:")
print(patient_risk_model_path)
print()
print("Metadata:")
print(metadata_path)


Patient Risk model saved successfully.

Model:
B:\VS CODE\TY\EDAI\TRAINED MODEL\models\patient_risk\patient_risk_model.joblib

Metadata:
B:\VS CODE\TY\EDAI\TRAINED MODEL\models\patient_risk\metadata.json


In [20]:
# ============================================================
# MODEL 2 — 30-DAY READMISSION PREDICTION
# ============================================================

print("MODEL 2 — READMISSION PREDICTION")
print("=" * 60)

TARGET_READMISSION = "readmitted_30d"

X_readmission = X_encoded.copy()
y_readmission = df[TARGET_READMISSION]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_readmission,
    y_readmission,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_readmission
)

print(f"Training records : {len(Xr_train):,}")
print(f"Testing records  : {len(Xr_test):,}")
print(f"Features         : {Xr_train.shape[1]}")
print(f"Positive cases   : {y_readmission.sum():,}")

readmission_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=14,
    min_samples_split=8,
    min_samples_leaf=3,
    class_weight="balanced",
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

readmission_model.fit(
    Xr_train,
    yr_train
)

print("\nReadmission model training completed.")

MODEL 2 — READMISSION PREDICTION
Training records : 40,000
Testing records  : 10,000
Features         : 29
Positive cases   : 15,861

Readmission model training completed.


In [21]:
# ============================================================
# MODEL 2 — READMISSION EVALUATION
# ============================================================

readmission_predictions = (
    readmission_model.predict(Xr_test)
)

readmission_probabilities = (
    readmission_model.predict_proba(Xr_test)[:, 1]
)

readmission_accuracy = accuracy_score(
    yr_test,
    readmission_predictions
)

readmission_auc = roc_auc_score(
    yr_test,
    readmission_probabilities
)

readmission_report = classification_report(
    yr_test,
    readmission_predictions,
    target_names=["Not Readmitted", "Readmitted"],
    digits=4
)

print("READMISSION MODEL RESULTS")
print("=" * 60)
print(f"Accuracy : {readmission_accuracy:.4f}")
print(f"ROC-AUC  : {readmission_auc:.4f}")

print("\nClassification Report")
print("-" * 60)
print(readmission_report)

print("\nConfusion Matrix")
print("-" * 60)

readmission_cm = confusion_matrix(
    yr_test,
    readmission_predictions
)

print(readmission_cm)

READMISSION MODEL RESULTS
Accuracy : 0.6855
ROC-AUC  : 0.7108

Classification Report
------------------------------------------------------------
                precision    recall  f1-score   support

Not Readmitted     0.7785    0.7540    0.7660      6828
    Readmitted     0.5040    0.5381    0.5205      3172

      accuracy                         0.6855     10000
     macro avg     0.6412    0.6461    0.6433     10000
  weighted avg     0.6914    0.6855    0.6881     10000


Confusion Matrix
------------------------------------------------------------
[[5148 1680]
 [1465 1707]]


In [22]:
# ============================================================
# MODEL 2 — FEATURE IMPORTANCE
# ============================================================

readmission_importance = pd.DataFrame({
    "feature": Xr_train.columns,
    "importance": readmission_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("Top Readmission Risk Features")
print("=" * 60)

display(
    readmission_importance.head(15)
)

Top Readmission Risk Features


,feature,importance
0,age,0.197634
2,systolic_bp,0.072188
4,temperature,0.071453
1,heart_rate,0.071387
3,diastolic_bp,0.070426
12,previous_emergency_visits,0.069400
11,previous_admissions,0.069339
6,respiratory_rate,0.068227
5,spo2,0.062501
13,medication_count,0.053165


In [23]:
# ============================================================
# SAVE MODEL 2 — READMISSION
# ============================================================

READMISSION_DIR = MODELS / "readmission"
READMISSION_DIR.mkdir(parents=True, exist_ok=True)

readmission_model_path = (
    READMISSION_DIR / "readmission_model.joblib"
)

joblib.dump(
    readmission_model,
    readmission_model_path
)

readmission_metadata = {
    "model_name": "ANVAYA 30-Day Readmission Model",
    "model_type": "RandomForestClassifier",
    "version": "1.0",
    "target": "readmitted_30d",
    "features": list(Xr_train.columns),
    "training_records": int(len(Xr_train)),
    "test_records": int(len(Xr_test)),
    "metrics": {
        "accuracy": round(readmission_accuracy, 4),
        "roc_auc": round(readmission_auc, 4)
    },
    "data_type": "synthetic_demo_data",
    "purpose": "Academic hospital readmission-risk prototype"
}

with open(
    READMISSION_DIR / "metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        readmission_metadata,
        f,
        indent=4
    )

print("Readmission model saved.")
print(readmission_model_path)

Readmission model saved.
B:\VS CODE\TY\EDAI\TRAINED MODEL\models\readmission\readmission_model.joblib


In [24]:
# ============================================================
# MODEL 3 — LENGTH OF STAY PREDICTION
# ============================================================

print("MODEL 3 — LENGTH OF STAY")
print("=" * 60)

TARGET_LOS = "length_of_stay"

X_los = X_encoded.copy()
y_los = df[TARGET_LOS]

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_los,
    y_los,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print(f"Training records : {len(Xl_train):,}")
print(f"Testing records  : {len(Xl_test):,}")
print(f"Features         : {Xl_train.shape[1]}")

los_model = RandomForestRegressor(
    n_estimators=400,
    max_depth=16,
    min_samples_split=8,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

los_model.fit(
    Xl_train,
    yl_train
)

print("\nLength-of-stay model training completed.")

MODEL 3 — LENGTH OF STAY
Training records : 40,000
Testing records  : 10,000
Features         : 29

Length-of-stay model training completed.


In [25]:
# ============================================================
# MODEL 3 — LENGTH OF STAY EVALUATION
# ============================================================

los_predictions = los_model.predict(
    Xl_test
)

los_mae = mean_absolute_error(
    yl_test,
    los_predictions
)

los_rmse = np.sqrt(
    mean_squared_error(
        yl_test,
        los_predictions
    )
)

los_r2 = r2_score(
    yl_test,
    los_predictions
)

print("LENGTH OF STAY MODEL RESULTS")
print("=" * 60)

print(f"MAE  : {los_mae:.4f} days")
print(f"RMSE : {los_rmse:.4f} days")
print(f"R²   : {los_r2:.4f}")

print("\nExample predictions:")

los_examples = pd.DataFrame({
    "Actual_LOS": yl_test.iloc[:10].values,
    "Predicted_LOS": np.round(
        los_predictions[:10],
        2
    )
})

display(los_examples)

LENGTH OF STAY MODEL RESULTS
MAE  : 1.2034 days
RMSE : 1.4963 days
R²   : 0.4856

Example predictions:


,Actual_LOS,Predicted_LOS
0,5.9,4.86
1,6.7,5.72
2,6.1,5.53
3,2.4,3.54
4,5.0,4.54
5,2.9,3.59
6,1.0,3.30
7,6.3,5.24
8,4.4,4.32
9,5.5,4.08


In [26]:
# ============================================================
# MODEL 3 — FEATURE IMPORTANCE
# ============================================================

los_importance = pd.DataFrame({
    "feature": Xl_train.columns,
    "importance": los_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("Top Length-of-Stay Features")
print("=" * 60)

display(
    los_importance.head(15)
)

Top Length-of-Stay Features


,feature,importance
0,age,0.177520
15,icu_flag,0.123852
9,cardiac_condition,0.121047
8,hypertension,0.092267
14,lab_abnormal_count,0.064659
11,previous_admissions,0.053412
10,respiratory_condition,0.052243
7,diabetes,0.049211
3,diastolic_bp,0.032651
2,systolic_bp,0.032473


In [27]:
# ============================================================
# SAVE MODEL 3 — LENGTH OF STAY
# ============================================================

LOS_DIR = MODELS / "length_of_stay"
LOS_DIR.mkdir(parents=True, exist_ok=True)

los_model_path = (
    LOS_DIR / "length_of_stay_model.joblib"
)

joblib.dump(
    los_model,
    los_model_path
)

los_metadata = {
    "model_name": "ANVAYA Length-of-Stay Prediction Model",
    "model_type": "RandomForestRegressor",
    "version": "1.0",
    "target": "length_of_stay",
    "features": list(Xl_train.columns),
    "training_records": int(len(Xl_train)),
    "test_records": int(len(Xl_test)),
    "metrics": {
        "mae_days": round(los_mae, 4),
        "rmse_days": round(los_rmse, 4),
        "r2": round(los_r2, 4)
    },
    "data_type": "synthetic_demo_data",
    "purpose": "Academic hospital length-of-stay prototype"
}

with open(
    LOS_DIR / "metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        los_metadata,
        f,
        indent=4
    )

print("Length-of-stay model saved.")
print(los_model_path)

Length-of-stay model saved.
B:\VS CODE\TY\EDAI\TRAINED MODEL\models\length_of_stay\length_of_stay_model.joblib


In [28]:
# ============================================================
# MODEL 4 — CLINICAL DETERIORATION
# ============================================================

print("MODEL 4 — CLINICAL DETERIORATION")
print("=" * 60)

TARGET_DETERIORATION = "deterioration"

X_det = X_encoded.copy()
y_det = df[TARGET_DETERIORATION]

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_det,
    y_det,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_det
)

print(f"Training records : {len(Xd_train):,}")
print(f"Testing records  : {len(Xd_test):,}")
print(f"Features         : {Xd_train.shape[1]}")
print(f"Positive cases   : {y_det.sum():,}")

deterioration_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=14,
    min_samples_split=8,
    min_samples_leaf=3,
    class_weight="balanced",
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

deterioration_model.fit(
    Xd_train,
    yd_train
)

print("\nDeterioration model training completed.")

MODEL 4 — CLINICAL DETERIORATION
Training records : 40,000
Testing records  : 10,000
Features         : 29
Positive cases   : 2,078

Deterioration model training completed.


In [29]:
# ============================================================
# MODEL 4 — DETERIORATION EVALUATION
# ============================================================

det_predictions = (
    deterioration_model.predict(Xd_test)
)

det_probabilities = (
    deterioration_model.predict_proba(Xd_test)[:, 1]
)

det_accuracy = accuracy_score(
    yd_test,
    det_predictions
)

det_auc = roc_auc_score(
    yd_test,
    det_probabilities
)

det_report = classification_report(
    yd_test,
    det_predictions,
    target_names=[
        "Stable",
        "Deterioration"
    ],
    digits=4
)

print("DETERIORATION MODEL RESULTS")
print("=" * 60)

print(f"Accuracy : {det_accuracy:.4f}")
print(f"ROC-AUC  : {det_auc:.4f}")

print("\nClassification Report")
print("-" * 60)
print(det_report)

print("\nConfusion Matrix")
print("-" * 60)

det_cm = confusion_matrix(
    yd_test,
    det_predictions
)

print(det_cm)

DETERIORATION MODEL RESULTS
Accuracy : 0.9575
ROC-AUC  : 0.6547

Classification Report
------------------------------------------------------------
               precision    recall  f1-score   support

       Stable     0.9590    0.9983    0.9783      9584
Deterioration     0.3043    0.0168    0.0319       416

     accuracy                         0.9575     10000
    macro avg     0.6317    0.5076    0.5051     10000
 weighted avg     0.9318    0.9575    0.9389     10000


Confusion Matrix
------------------------------------------------------------
[[9568   16]
 [ 409    7]]


In [30]:
# ============================================================
# SAVE MODEL 4 — DETERIORATION
# ============================================================

DETERIORATION_DIR = MODELS / "deterioration"
DETERIORATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

deterioration_model_path = (
    DETERIORATION_DIR / "deterioration_model.joblib"
)

joblib.dump(
    deterioration_model,
    deterioration_model_path
)

deterioration_metadata = {
    "model_name": "ANVAYA Clinical Deterioration Model",
    "model_type": "RandomForestClassifier",
    "version": "1.0",
    "target": "deterioration",
    "features": list(Xd_train.columns),
    "training_records": int(len(Xd_train)),
    "test_records": int(len(Xd_test)),
    "metrics": {
        "accuracy": round(det_accuracy, 4),
        "roc_auc": round(det_auc, 4)
    },
    "data_type": "synthetic_demo_data",
    "purpose": "Academic clinical deterioration-risk prototype"
}

with open(
    DETERIORATION_DIR / "metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        deterioration_metadata,
        f,
        indent=4
    )

print("Deterioration model saved.")
print(deterioration_model_path)

Deterioration model saved.
B:\VS CODE\TY\EDAI\TRAINED MODEL\models\deterioration\deterioration_model.joblib


In [31]:
# ============================================================
# MODEL 5 — BED OCCUPANCY FORECASTING
# ============================================================

print("MODEL 5 — BED OCCUPANCY FORECAST")
print("=" * 60)

OCCUPANCY_DAYS = 365

occupancy_rng = np.random.default_rng(
    RANDOM_STATE + 100
)

dates = pd.date_range(
    start="2025-01-01",
    periods=OCCUPANCY_DAYS,
    freq="D"
)

TOTAL_BEDS = 168

occupancy_rows = []

current_occupancy = 105

for i, date in enumerate(dates):

    day_of_week = date.dayofweek
    month = date.month

    emergency_admissions = max(
        0,
        int(
            occupancy_rng.poisson(
                8
                + (2 if day_of_week >= 5 else 0)
            )
        )
    )

    elective_admissions = max(
        0,
        int(
            occupancy_rng.poisson(12)
        )
    )

    discharges = max(
        0,
        int(
            occupancy_rng.poisson(14)
        )
    )

    average_los = np.clip(
        occupancy_rng.normal(
            4.8,
            0.7
        ),
        2.5,
        8
    )

    # Seasonal operational pressure
    seasonal_factor = (
        5
        if month in [6, 7, 8]
        else 0
    )

    occupancy_change = (
        emergency_admissions
        + elective_admissions
        - discharges
        + seasonal_factor * 0.15
    )

    noise = occupancy_rng.normal(
        0,
        3
    )

    next_occupancy = np.clip(
        current_occupancy
        + occupancy_change
        + noise,
        50,
        TOTAL_BEDS
    )

    occupancy_rows.append({
        "date": date,
        "current_occupancy": current_occupancy,
        "emergency_admissions": emergency_admissions,
        "elective_admissions": elective_admissions,
        "discharges": discharges,
        "average_los": round(
            average_los,
            2
        ),
        "day_of_week": day_of_week,
        "month": month,
        "total_beds": TOTAL_BEDS,
        "next_day_occupancy": round(
            next_occupancy,
            2
        )
    })

    current_occupancy = next_occupancy

occupancy_df = pd.DataFrame(
    occupancy_rows
)

occupancy_df["current_occupancy_rate"] = (
    occupancy_df["current_occupancy"]
    / TOTAL_BEDS
)

occupancy_df["next_day_occupancy_rate"] = (
    occupancy_df["next_day_occupancy"]
    / TOTAL_BEDS
)

print("Occupancy dataset generated.")
print(f"Days: {len(occupancy_df):,}")

display(
    occupancy_df.head()
)

MODEL 5 — BED OCCUPANCY FORECAST
Occupancy dataset generated.
Days: 365


,date,current_occupancy,emergency_admissions,elective_admissions,discharges,average_los,day_of_week,month,total_beds,next_day_occupancy,current_occupancy_rate,next_day_occupancy_rate
0,2025-01-01,105.000000,6,9,13,4.39,2,1,168,104.47,0.625000,0.621845
1,2025-01-02,104.467349,9,12,14,5.13,3,1,168,115.10,0.621829,0.685119
2,2025-01-03,115.096938,6,12,10,4.95,4,1,168,127.17,0.685101,0.756964
3,2025-01-04,127.166234,14,13,16,5.66,5,1,168,137.15,0.756942,0.816369
4,2025-01-05,137.145530,11,15,13,4.07,6,1,168,152.31,0.816342,0.906607


In [32]:
# ============================================================
# MODEL 5 — OCCUPANCY TRAINING
# ============================================================

OCCUPANCY_FEATURES = [
    "current_occupancy",
    "emergency_admissions",
    "elective_admissions",
    "discharges",
    "average_los",
    "day_of_week",
    "month",
    "total_beds",
    "current_occupancy_rate"
]

X_occ = occupancy_df[
    OCCUPANCY_FEATURES
]

y_occ = occupancy_df[
    "next_day_occupancy_rate"
]

# Chronological split — no random shuffle
split_index = int(
    len(occupancy_df) * 0.80
)

Xo_train = X_occ.iloc[
    :split_index
].copy()

Xo_test = X_occ.iloc[
    split_index:
].copy()

yo_train = y_occ.iloc[
    :split_index
].copy()

yo_test = y_occ.iloc[
    split_index:
].copy()

print("Occupancy forecasting split")
print("=" * 60)
print(f"Training days : {len(Xo_train)}")
print(f"Testing days  : {len(Xo_test)}")

occupancy_model = RandomForestRegressor(
    n_estimators=400,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

occupancy_model.fit(
    Xo_train,
    yo_train
)

print("\nOccupancy model training completed.")

Occupancy forecasting split
Training days : 292
Testing days  : 73

Occupancy model training completed.


In [33]:
# ============================================================
# MODEL 5 — OCCUPANCY EVALUATION
# ============================================================

occupancy_predictions = (
    occupancy_model.predict(Xo_test)
)

occupancy_mae = mean_absolute_error(
    yo_test,
    occupancy_predictions
)

occupancy_rmse = np.sqrt(
    mean_squared_error(
        yo_test,
        occupancy_predictions
    )
)

occupancy_r2 = r2_score(
    yo_test,
    occupancy_predictions
)

print("BED OCCUPANCY FORECAST RESULTS")
print("=" * 60)

print(
    f"MAE  : {occupancy_mae:.4f}"
)

print(
    f"RMSE : {occupancy_rmse:.4f}"
)

print(
    f"R²   : {occupancy_r2:.4f}"
)

print("\nExample forecasts:")

occupancy_examples = pd.DataFrame({
    "Actual_Next_Day_%": (
        yo_test.iloc[:10].values * 100
    ).round(2),

    "Predicted_Next_Day_%": (
        occupancy_predictions[:10] * 100
    ).round(2)
})

display(
    occupancy_examples
)

BED OCCUPANCY FORECAST RESULTS
MAE  : 0.0043
RMSE : 0.0079
R²   : 0.3153

Example forecasts:


,Actual_Next_Day_%,Predicted_Next_Day_%
0,97.90,99.66
1,95.53,99.06
2,99.99,98.96
3,100.00,99.86
4,100.00,99.91
5,100.00,100.00
6,100.00,100.00
7,97.19,98.88
8,96.41,98.12
9,100.00,99.64


In [34]:
# ============================================================
# SAVE MODEL 5 — BED OCCUPANCY
# ============================================================

OCCUPANCY_DIR = MODELS / "bed_occupancy"

OCCUPANCY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

occupancy_model_path = (
    OCCUPANCY_DIR / "bed_occupancy_model.joblib"
)

joblib.dump(
    occupancy_model,
    occupancy_model_path
)

occupancy_metadata = {
    "model_name": "ANVAYA Bed Occupancy Forecast Model",
    "model_type": "RandomForestRegressor",
    "version": "1.0",
    "target": "next_day_occupancy_rate",
    "features": OCCUPANCY_FEATURES,
    "training_days": int(len(Xo_train)),
    "test_days": int(len(Xo_test)),
    "total_beds": TOTAL_BEDS,
    "metrics": {
        "mae": round(occupancy_mae, 4),
        "rmse": round(occupancy_rmse, 4),
        "r2": round(occupancy_r2, 4)
    },
    "data_type": "synthetic_demo_data",
    "purpose": "Academic hospital capacity forecasting prototype"
}

with open(
    OCCUPANCY_DIR / "metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        occupancy_metadata,
        f,
        indent=4
    )

print("Bed occupancy model saved.")
print(occupancy_model_path)

Bed occupancy model saved.
B:\VS CODE\TY\EDAI\TRAINED MODEL\models\bed_occupancy\bed_occupancy_model.joblib


In [35]:
# ============================================================
# SAVE OCCUPANCY DATASET
# ============================================================

occupancy_path = (
    DATA_RAW / "hospital_daily_occupancy.csv"
)

occupancy_df.to_csv(
    occupancy_path,
    index=False
)

print("Occupancy dataset saved:")
print(occupancy_path)

Occupancy dataset saved:
B:\VS CODE\TY\EDAI\TRAINED MODEL\data\raw\hospital_daily_occupancy.csv


In [36]:
# ============================================================
# SAVE MAIN ML DATASET
# ============================================================

main_dataset_path = (
    DATA_RAW / "anvaya_hospital_ml_dataset.csv"
)

df.to_csv(
    main_dataset_path,
    index=False
)

print("Main hospital dataset saved:")
print(main_dataset_path)

Main hospital dataset saved:
B:\VS CODE\TY\EDAI\TRAINED MODEL\data\raw\anvaya_hospital_ml_dataset.csv


In [37]:
# ============================================================
# ANVAYA MODEL REGISTRY
# ============================================================

model_registry = {
    "project": "ANVAYA SAṂHATI",
    "pipeline_version": "1.0",
    "data_type": "synthetic_demo_data",

    "models": {
        "patient_risk": {
            "file": "models/patient_risk/patient_risk_model.joblib",
            "type": "classification",
            "target": "high_risk",
            "threshold": BEST_THRESHOLD,
            "roc_auc": round(final_auc, 4)
        },

        "readmission": {
            "file": "models/readmission/readmission_model.joblib",
            "type": "classification",
            "target": "readmitted_30d",
            "roc_auc": round(readmission_auc, 4)
        },

        "length_of_stay": {
            "file": "models/length_of_stay/length_of_stay_model.joblib",
            "type": "regression",
            "target": "length_of_stay",
            "mae_days": round(los_mae, 4),
            "r2": round(los_r2, 4)
        },

        "deterioration": {
            "file": "models/deterioration/deterioration_model.joblib",
            "type": "classification",
            "target": "deterioration",
            "roc_auc": round(det_auc, 4)
        },

        "bed_occupancy": {
            "file": "models/bed_occupancy/bed_occupancy_model.joblib",
            "type": "regression",
            "target": "next_day_occupancy_rate",
            "mae": round(occupancy_mae, 4),
            "r2": round(occupancy_r2, 4)
        }
    }
}

registry_path = (
    MODELS / "model_registry.json"
)

with open(
    registry_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        model_registry,
        f,
        indent=4
    )

print("Model registry created.")
print(registry_path)

Model registry created.
B:\VS CODE\TY\EDAI\TRAINED MODEL\models\model_registry.json


In [38]:
# ============================================================
# ANVAYA ML — FINAL MODEL VERIFICATION
# ============================================================

print()
print("=" * 75)
print("             ANVAYA SAṂHATI — ML PIPELINE")
print("                  FINAL VERIFICATION")
print("=" * 75)

print()

print("MODEL 1 — PATIENT RISK")
print("-" * 75)
print(f"ROC-AUC            : {final_auc:.4f}")
print(f"Decision Threshold : {BEST_THRESHOLD:.2f}")

print()

print("MODEL 2 — 30-DAY READMISSION")
print("-" * 75)
print(f"ROC-AUC            : {readmission_auc:.4f}")
print(f"Accuracy            : {readmission_accuracy:.4f}")

print()

print("MODEL 3 — LENGTH OF STAY")
print("-" * 75)
print(f"MAE                : {los_mae:.4f} days")
print(f"RMSE               : {los_rmse:.4f} days")
print(f"R²                 : {los_r2:.4f}")

print()

print("MODEL 4 — CLINICAL DETERIORATION")
print("-" * 75)
print(f"ROC-AUC            : {det_auc:.4f}")
print(f"Accuracy            : {det_accuracy:.4f}")

print()

print("MODEL 5 — BED OCCUPANCY")
print("-" * 75)
print(f"MAE                : {occupancy_mae:.4f}")
print(f"RMSE               : {occupancy_rmse:.4f}")
print(f"R²                 : {occupancy_r2:.4f}")

print()
print("=" * 75)
print("MODEL ARTIFACTS")
print("=" * 75)

for path in [
    patient_risk_model_path,
    readmission_model_path,
    los_model_path,
    deterioration_model_path,
    occupancy_model_path,
    registry_path
]:
    print(f"✓ {path}")

print()
print("=" * 75)
print("ALL ANVAYA ML MODELS TRAINED AND SAVED")
print("=" * 75)


             ANVAYA SAṂHATI — ML PIPELINE
                  FINAL VERIFICATION

MODEL 1 — PATIENT RISK
---------------------------------------------------------------------------
ROC-AUC            : 0.7433
Decision Threshold : 0.42

MODEL 2 — 30-DAY READMISSION
---------------------------------------------------------------------------
ROC-AUC            : 0.7108
Accuracy            : 0.6855

MODEL 3 — LENGTH OF STAY
---------------------------------------------------------------------------
MAE                : 1.2034 days
RMSE               : 1.4963 days
R²                 : 0.4856

MODEL 4 — CLINICAL DETERIORATION
---------------------------------------------------------------------------
ROC-AUC            : 0.6547
Accuracy            : 0.9575

MODEL 5 — BED OCCUPANCY
---------------------------------------------------------------------------
MAE                : 0.0043
RMSE               : 0.0079
R²                 : 0.3153

MODEL ARTIFACTS
✓ B:\VS CODE\TY\EDAI\TRAINED MODEL\mod

In [39]:
# ============================================================
# ANVAYA SAṂHATI — LIVE ML DEMONSTRATION
# ============================================================

print("=" * 75)
print("             ANVAYA SAṂHATI — LIVE ML DEMO")
print("=" * 75)

# ------------------------------------------------------------
# 1. LOAD SAVED MODELS
# ------------------------------------------------------------

loaded_patient_risk = joblib.load(
    patient_risk_model_path
)

loaded_readmission = joblib.load(
    readmission_model_path
)

loaded_los = joblib.load(
    los_model_path
)

loaded_deterioration = joblib.load(
    deterioration_model_path
)

loaded_occupancy = joblib.load(
    occupancy_model_path
)

print("\n✓ All 5 trained models loaded successfully.")

# ------------------------------------------------------------
# 2. CREATE A NEW PATIENT
# ------------------------------------------------------------

demo_patient = pd.DataFrame([{
    "age": 68,
    "gender": "Male",
    "department": "Cardiology",
    "encounter_type": "EMERGENCY",

    "heart_rate": 108.0,
    "systolic_bp": 148.0,
    "diastolic_bp": 88.0,
    "temperature": 100.4,
    "spo2": 92.0,
    "respiratory_rate": 24.0,

    "diabetes": 1,
    "hypertension": 1,
    "cardiac_condition": 1,
    "respiratory_condition": 0,

    "previous_admissions": 3,
    "previous_emergency_visits": 2,
    "medication_count": 7,
    "lab_abnormal_count": 4,
    "icu_flag": 0
}])

# ------------------------------------------------------------
# 3. APPLY SAME ENCODING USED DURING TRAINING
# ------------------------------------------------------------

demo_encoded = pd.get_dummies(
    demo_patient,
    columns=[
        "gender",
        "department",
        "encounter_type"
    ],
    dtype=int
)

# Make sure the new patient has EXACTLY the training columns
demo_encoded = demo_encoded.reindex(
    columns=X_encoded.columns,
    fill_value=0
)

# ------------------------------------------------------------
# 4. PATIENT RISK
# ------------------------------------------------------------

risk_probability = (
    loaded_patient_risk
    .predict_proba(demo_encoded)[0, 1]
)

risk_prediction = (
    risk_probability >= BEST_THRESHOLD
)

# ------------------------------------------------------------
# 5. READMISSION
# ------------------------------------------------------------

readmission_probability = (
    loaded_readmission
    .predict_proba(demo_encoded)[0, 1]
)

# ------------------------------------------------------------
# 6. LENGTH OF STAY
# ------------------------------------------------------------

predicted_los = (
    loaded_los
    .predict(demo_encoded)[0]
)

# ------------------------------------------------------------
# 7. DETERIORATION
# ------------------------------------------------------------

deterioration_probability = (
    loaded_deterioration
    .predict_proba(demo_encoded)[0, 1]
)

# ------------------------------------------------------------
# 8. DISPLAY PATIENT
# ------------------------------------------------------------

print("\n")
print("DEMO PATIENT")
print("-" * 75)

print("Age                 :", demo_patient["age"].iloc[0])
print("Gender              :", demo_patient["gender"].iloc[0])
print("Department          :", demo_patient["department"].iloc[0])
print("Encounter           :", demo_patient["encounter_type"].iloc[0])

print("\nVital Signs")
print("-" * 75)

print("Heart Rate          :", demo_patient["heart_rate"].iloc[0])
print("Blood Pressure      :",
      f"{demo_patient['systolic_bp'].iloc[0]:.0f}/"
      f"{demo_patient['diastolic_bp'].iloc[0]:.0f}")

print("Temperature         :",
      demo_patient["temperature"].iloc[0])

print("SpO2                :",
      demo_patient["spo2"].iloc[0])

print("Respiratory Rate    :",
      demo_patient["respiratory_rate"].iloc[0])

# ------------------------------------------------------------
# 9. AI PREDICTIONS
# ------------------------------------------------------------

print("\n")
print("AI / ML PREDICTIONS")
print("=" * 75)

print(
    f"Patient Risk        : "
    f"{'HIGH' if risk_prediction else 'LOW'}"
)

print(
    f"Risk Probability    : "
    f"{risk_probability * 100:.2f}%"
)

print(
    f"Readmission Risk    : "
    f"{readmission_probability * 100:.2f}%"
)

print(
    f"Predicted LOS       : "
    f"{predicted_los:.2f} days"
)

print(
    f"Deterioration Risk  : "
    f"{deterioration_probability * 100:.2f}%"
)

print("\n")
print("=" * 75)
print("MODEL STATUS")
print("=" * 75)

print("✓ Patient Risk Model")
print("✓ Readmission Model")
print("✓ Length-of-Stay Model")
print("✓ Clinical Deterioration Model")
print("✓ Bed Occupancy Model")

print("\n✓ Models loaded from saved .joblib artifacts")
print("✓ New patient processed successfully")
print("✓ Predictions generated successfully")

print("\nNOTE:")
print("This demonstration uses synthetic data.")
print("Predictions are for academic prototype demonstration only.")

print("\n")
print("=" * 75)
print("              ANVAYA ML DEMONSTRATION COMPLETE")
print("=" * 75)

             ANVAYA SAṂHATI — LIVE ML DEMO

✓ All 5 trained models loaded successfully.


DEMO PATIENT
---------------------------------------------------------------------------
Age                 : 68
Gender              : Male
Department          : Cardiology
Encounter           : EMERGENCY

Vital Signs
---------------------------------------------------------------------------
Heart Rate          : 108.0
Blood Pressure      : 148/88
Temperature         : 100.4
SpO2                : 92.0
Respiratory Rate    : 24.0


AI / ML PREDICTIONS
Patient Risk        : HIGH
Risk Probability    : 48.72%
Readmission Risk    : 71.63%
Predicted LOS       : 8.48 days
Deterioration Risk  : 24.19%


MODEL STATUS
✓ Patient Risk Model
✓ Readmission Model
✓ Length-of-Stay Model
✓ Clinical Deterioration Model
✓ Bed Occupancy Model

✓ Models loaded from saved .joblib artifacts
✓ New patient processed successfully
✓ Predictions generated successfully

NOTE:
This demonstration uses synthetic data.
Predictio